# Inspect Mayabeque Soil Dataset

This notebook performs an initial inspection of the 1:25,000 soil map
used in the GeoSoil-Cuba project.

The source dataset is a MapInfo dataset consisting of a `.TAB` file
and its associated data/index files. The inspection verifies that the
source files are present, loads the dataset with GeoPandas, examines
its spatial and attribute structure, checks for missing values, and
produces an initial map.

This notebook does not modify the source dataset. Data preparation and
analysis are performed in subsequent notebooks.

## 1. Setup

In [ ]:
from pathlib import Path

import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt


# Project directories
PROJECT_DIR = Path("..").resolve()
DATA_DIR = PROJECT_DIR / "data" / "raw" / "Mapa_Suelos_25000"
OUTPUT_DIR = PROJECT_DIR / "outputs" / "maps"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# Source dataset
DATASET_NAME = "24 Myb suelos 25 mil"
TAB_FILE = DATA_DIR / f"{DATASET_NAME}.TAB"

## 2. Verify source dataset files

In [ ]:
extensions = [".DAT", ".ID", ".MAP", ".TAB", ".IND"]

files = {
    ext: DATA_DIR / f"{DATASET_NAME}{ext}"
    for ext in extensions
}

file_info = []

for ext, path in files.items():
    file_info.append({
        "Extension": ext,
        "File": path.name,
        "Exists": path.exists(),
        "Size_MB": round(path.stat().st_size / (1024 * 1024), 3)
        if path.exists() else None
    })

file_info_df = pd.DataFrame(file_info)

display(file_info_df)

## 3. Load soil dataset

In [ ]:
gdf = gpd.read_file(TAB_FILE)

print("Dataset successfully loaded.")
print()
print("Number of features:", len(gdf))
print("Number of columns:", len(gdf.columns))

print("Columns:")
for column in gdf.columns:
    print(f"  {column}")

display(gdf.head())

print("Geometry type(s):")
print(gdf.geometry.geom_type.value_counts())

print("\nCRS:")
print(gdf.crs)

print("\nBounds:")
print(gdf.total_bounds)

## 4. Inspect structure and attributes

In [ ]:
print(f"Features: {len(gdf):,}")
print(f"Columns: {len(gdf.columns)}")
print(f"CRS: {gdf.crs}")
print(f"Geometry types: {gdf.geometry.geom_type.unique()}")

west, south, east, north = gdf.total_bounds

print("\nDataset extent:")
print(f"West:  {west}")
print(f"South: {south}")
print(f"East:  {east}")
print(f"North: {north}")

## 5. Check missing values

In [ ]:
missing = (
    gdf.isna()
    .sum()
    .sort_values(ascending=False)
)

display(missing.to_frame("Missing values"))

## 6. Create initial soil map

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt


OUTPUT_FILE = OUTPUT_DIR / "mayabeque_soil_map_25000.png"

fig, ax = plt.subplots(figsize=(12, 10))

gdf.plot(
    ax=ax,
    edgecolor="black",
    linewidth=0.3
)

ax.set_title("Mayabeque Soil Map — 1:25,000")
ax.set_xlabel("X")
ax.set_ylabel("Y")

fig.savefig(
    OUTPUT_FILE,
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

print(f"Saved: {OUTPUT_FILE}")